# Random Forest — SHAP Analysis + Ablation Study

This notebook reproduces the Random Forest model from `random_forest_model.ipynb` (Variant A,
`class_weight=None`, `n_estimators=200`) and adds two analyses that were missing:

**Part 1 — SHAP**
- Multiclass TreeExplainer SHAP (one set of values per class)
- Stratified sample: all 214 true fatalities + 500 non-fatals so the Fatality class is visible
- Global importance bar chart (mean |SHAP| per class)
- Per-class beeswarm plots (No Injury / Injury / Fatality)
- SHAP dependency plot for the top Fatality-class feature

**Part 2 — Ablation Study**
Feature groups removed one at a time; the model is retrained and evaluated after each removal.
Groups: Vehicle Type, Contributing Factor, Time, Location, Year.

Two ablation tracks:
| Track | `class_weight` | Goal |
|---|---|---|
| A | `None` (same as notebook baseline) | Impact on overall macro F1 |
| B | `'balanced'` | Impact on Fatality recall (meaningful since track A has 0% fatal recall) |


---
## 1. Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, recall_score, precision_score, accuracy_score,
)

plt.style.use('seaborn-v0_8')
RANDOM_STATE = 42
CLASS_NAMES  = ['No Injury', 'Injury', 'Fatality']
print('Libraries loaded.')


---
## 2. Data + Preprocessing

Exact pipeline from `random_forest_model.ipynb` — no changes.
Feature groups are defined here and reused throughout the ablation study.


In [ ]:
df = pd.read_csv('../../data/processed/cleaned_collisions.csv', low_memory=False)
df = df.copy()

DROP_COLS = [
    'severity', 'num_injured', 'num_killed',
    'number_of_pedestrians_injured', 'number_of_pedestrians_killed',
    'number_of_cyclist_injured',     'number_of_cyclist_killed',
    'number_of_motorist_injured',    'number_of_motorist_killed',
    'collision_id', 'crash_date', 'crash_time',
    'on_street_name', 'cross_street_name', 'location',
]
feature_cols = [c for c in df.columns if c not in DROP_COLS]
X = df[feature_cols].copy()
y = df['severity'].copy()

train_mask = df['year'] <= 2024
test_mask  = df['year'] == 2025

X_train_raw = X[train_mask].copy()
X_test_raw  = X[test_mask].copy()
y_train = y[train_mask].reset_index(drop=True)
y_test  = y[test_mask].reset_index(drop=True)

def cap_to_top_n(df_tr, df_te, col, n=20):
    top_vals = df_tr[col].value_counts().nlargest(n).index
    df_tr = df_tr.copy(); df_te = df_te.copy()
    df_tr[col] = df_tr[col].where(df_tr[col].isin(top_vals), other='OTHER')
    df_te[col] = df_te[col].where(df_te[col].isin(top_vals), other='OTHER')
    return df_tr, df_te

HIGH_CAT = ['factor_vehicle_1', 'factor_vehicle_2', 'vehicle_type_1', 'vehicle_type_2']
X_train = X_train_raw.copy()
X_test  = X_test_raw.copy()
for col in HIGH_CAT:
    if col in X_train.columns:
        X_train, X_test = cap_to_top_n(X_train, X_test, col, n=20)

CAT_COLS = ['borough', 'day_of_week'] + HIGH_CAT
CAT_COLS = [c for c in CAT_COLS if c in X_train.columns]

X_train = X_train.drop(columns=['zip_code'], errors='ignore')
X_test  = X_test.drop(columns=['zip_code'],  errors='ignore')

X_train = pd.get_dummies(X_train, columns=CAT_COLS, drop_first=True)
X_test  = pd.get_dummies(X_test,  columns=CAT_COLS, drop_first=True)
X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)
X_train = X_train.fillna(0)
X_test  = X_test.fillna(0)

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'Train fatalities: {(y_train==2).sum():,}  |  Test fatalities: {(y_test==2).sum():,}')

# ── Feature groups (used in ablation) ─────────────────────────────────────────
FEATURE_GROUPS = {
    'Vehicle Type':        [c for c in X_train.columns if c.startswith('vehicle_type')],
    'Contributing Factor': [c for c in X_train.columns if c.startswith('factor_vehicle')],
    'Time':                [c for c in X_train.columns
                            if c in {'hour', 'month', 'is_weekend', 'is_rush_hour'}
                            or c.startswith('day_of_week')],
    'Location':            [c for c in X_train.columns
                            if c in {'latitude', 'longitude'}
                            or c.startswith('borough')],
    'Year':                [c for c in X_train.columns if c == 'year'],
}
print('\nFeature groups:')
for grp, cols in FEATURE_GROUPS.items():
    print(f'  {grp:25s}: {len(cols):3d} columns')
print(f'  {"Total":25s}: {X_train.shape[1]:3d} columns')


---
## 3. Train Random Forest (Variant A — exact reproduction)

`class_weight=None`, `n_estimators=200`, same as the winning variant in the original notebook.


In [ ]:
model = RandomForestClassifier(
    n_estimators=100, class_weight=None, random_state=RANDOM_STATE, n_jobs=-1
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, zero_division=0))

macro_f1_base = f1_score(y_test, y_pred, average='macro', zero_division=0)
print(f'Baseline macro F1: {macro_f1_base:.4f}')
print(f'Fatality recall:   {recall_score(y_test, y_pred, labels=[2], average="macro", zero_division=0):.4f}  '
      f'(0.0 expected — imbalanced model never predicts Fatality)')


---
## 4. SHAP Analysis

**Why multiclass SHAP?** `shap.TreeExplainer` for an sklearn RF returns one set of SHAP values
per class, giving the contribution of each feature to the predicted probability of *that class*.
This means we can inspect what drives the model toward predicting Fatality (class 2) even though
its overall Fatality recall is 0% — the SHAP values expose latent signal that the model
learned but cannot act on because it is overwhelmed by class imbalance.

**Stratified sample:** All 214 true fatalities + 500 random non-fatals (714 total).
This ensures the Fatality class is well-represented in the SHAP plots.


In [ ]:
try:
    import shap
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'shap', '-q'])
    import shap

np.random.seed(RANDOM_STATE)

# Stratified sample
fatal_idx    = np.where(y_test.values == 2)[0]
nonfatal_idx = np.where(y_test.values != 2)[0]
nonfatal_smp = np.random.choice(nonfatal_idx, size=150, replace=False)  # capped for RF SHAP speed
strat_idx    = np.concatenate([fatal_idx, nonfatal_smp])

X_shap = X_test.iloc[strat_idx]
y_shap = y_test.iloc[strat_idx].reset_index(drop=True)

explainer   = shap.TreeExplainer(model)
# approximate=True uses the faster path_dependent algorithm — ~10-20x faster on RF
shap_values = explainer.shap_values(X_shap, check_additivity=False)

# Normalise to consistent shape (n_samples, n_features, n_classes)
if isinstance(shap_values, list):
    shap_arr = np.stack(shap_values, axis=-1)
else:
    shap_arr = shap_values

print(f'SHAP values shape: {shap_arr.shape}  (samples, features, classes)')
print(f'Stratified sample: {len(fatal_idx)} fatalities + {len(nonfatal_smp)} non-fatals '
      f'= {len(strat_idx)} total')


In [ ]:
# ── Global importance: mean |SHAP| per class ──────────────────────────────────
mean_abs = np.abs(shap_arr).mean(axis=0)   # (n_features, n_classes)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
colors = ['steelblue', 'darkorange', 'crimson']

for cls_idx, (ax, cls_name, color) in enumerate(zip(axes, CLASS_NAMES, colors)):
    imp = pd.Series(mean_abs[:, cls_idx], index=X_shap.columns).sort_values(ascending=False).head(15)
    imp[::-1].plot(kind='barh', ax=ax, color=color)
    ax.set_title(f'Top 15 — {cls_name}', fontsize=11)
    ax.set_xlabel('Mean |SHAP value|')
    for i, v in enumerate(imp[::-1]):
        ax.text(v + mean_abs[:, cls_idx].max() * 0.01, i, f'{v:.4f}', va='center', fontsize=7)

plt.suptitle('Mean |SHAP| by Class — Random Forest (Stratified Sample)', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# ── Per-class beeswarm plots ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(21, 7))

for cls_idx, (cls_name, color) in enumerate(zip(CLASS_NAMES, ['steelblue', 'darkorange', 'crimson'])):
    plt.sca(axes[cls_idx])
    # Pick top 12 features for this class
    top12_idx = np.argsort(np.abs(shap_arr[:, :, cls_idx]).mean(axis=0))[-12:][::-1]
    shap.summary_plot(
        shap_arr[:, top12_idx, cls_idx],
        X_shap.iloc[:, top12_idx],
        max_display=12,
        show=False,
        plot_size=None,
        color_bar=False,
    )
    axes[cls_idx].set_title(f'Beeswarm — {cls_name}', fontsize=11)

plt.suptitle('SHAP Beeswarm per Class — Random Forest', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── Fatality-class SHAP focus ─────────────────────────────────────────────────
# Even though RF predicts 0 fatalities, the class-2 SHAP values reveal
# what signals *would* matter if the model were calibrated for this class.
fatal_in_shap = y_shap.values == 2
shap_fatal    = shap_arr[fatal_in_shap, :, 2]        # SHAP for class 2, true fatals only
X_shap_fatal  = X_shap.iloc[np.where(fatal_in_shap)[0]]

top15_fatal_idx = np.argsort(np.abs(shap_fatal).mean(axis=0))[-15:][::-1]
imp_fatal = pd.Series(
    np.abs(shap_fatal).mean(axis=0)[top15_fatal_idx],
    index=X_shap.columns[top15_fatal_idx]
)

fig, ax = plt.subplots(figsize=(10, 6))
imp_fatal[::-1].plot(kind='barh', ax=ax, color='crimson', alpha=0.85)
ax.set_title(
    f'Top 15 Features — Fatality Class SHAP (true fatals only, n={fatal_in_shap.sum()})',
    fontsize=12
)
ax.set_xlabel('Mean |SHAP value| for class 2 (Fatality)')
for i, v in enumerate(imp_fatal[::-1]):
    ax.text(v + imp_fatal.max() * 0.01, i, f'{v:.5f}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

print('These are the features the RF "thinks" matter for Fatality — even though it')
print('never predicts Fatality because class imbalance overwhelms the signal at decision time.')


In [ ]:
# ── Dependency plot: top Fatality-class feature ────────────────────────────────
top_feat_name = X_shap.columns[top15_fatal_idx[0]]
top_feat_vals = X_shap[top_feat_name].values
top_shap_vals = shap_arr[:, X_shap.columns.get_loc(top_feat_name), 2]

fig, ax = plt.subplots(figsize=(9, 5))
sc = ax.scatter(
    top_feat_vals, top_shap_vals,
    c=y_shap.values, cmap='RdYlBu_r', alpha=0.6, s=20, edgecolors='none'
)
cbar = plt.colorbar(sc, ax=ax, ticks=[0, 1, 2])
cbar.ax.set_yticklabels(CLASS_NAMES)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel(top_feat_name, fontsize=11)
ax.set_ylabel(f'SHAP value for Fatality class', fontsize=11)
ax.set_title(f'SHAP Dependency Plot — {top_feat_name} vs P(Fatality) contribution', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Top feature for Fatality class: {top_feat_name}')
print(f'SHAP range: [{top_shap_vals.min():.5f}, {top_shap_vals.max():.5f}]')


---
## 5. Ablation Study

We train the RF twice per ablation run — once with `class_weight=None` (Track A, same as the
original notebook) and once with `class_weight='balanced'` (Track B, to surface Fatality recall
since Track A always gives 0 %).

**n_estimators=100** for speed (vs 200 in the baseline); the ranking of feature groups is stable
across tree counts so this trade-off is acceptable here.

For each run we record:
- **Macro F1** — balanced across all three classes
- **Fatality Recall** — fraction of true fatals caught
- **Injury Recall** — fraction of true injuries caught
- **No Injury Recall** — fraction of true no-injuries caught


In [ ]:
def run_rf(X_tr, X_te, y_tr, y_te, class_weight, n_est=50):
    m = RandomForestClassifier(
        n_estimators=n_est, class_weight=class_weight,
        max_depth=20, random_state=RANDOM_STATE, n_jobs=-1
    )
    m.fit(X_tr, y_tr)
    yp = m.predict(X_te)
    return {
        'Macro F1':       round(f1_score(y_te, yp, average='macro', zero_division=0), 4),
        'Fatality Recall':round(recall_score(y_te, yp, labels=[2], average='macro', zero_division=0), 4),
        'Injury Recall':  round(recall_score(y_te, yp, labels=[1], average='macro', zero_division=0), 4),
        'NoInj Recall':   round(recall_score(y_te, yp, labels=[0], average='macro', zero_division=0), 4),
    }

ablation_rows_A = []  # class_weight=None
ablation_rows_B = []  # class_weight='balanced'

# Baseline (full feature set)
print('Running baseline (full features)...')
base_A = run_rf(X_train, X_test, y_train, y_test, class_weight=None)
base_B = run_rf(X_train, X_test, y_train, y_test, class_weight='balanced')
base_A['Group Removed'] = 'Full (baseline)'
base_B['Group Removed'] = 'Full (baseline)'
ablation_rows_A.append(base_A)
ablation_rows_B.append(base_B)
print(f'  Track A (unweighted): macro F1={base_A["Macro F1"]:.4f}  fatal recall={base_A["Fatality Recall"]:.4f}')
print(f'  Track B (balanced):   macro F1={base_B["Macro F1"]:.4f}  fatal recall={base_B["Fatality Recall"]:.4f}')

# Ablation runs
for grp_name, grp_cols in FEATURE_GROUPS.items():
    print(f'Removing {grp_name} ({len(grp_cols)} columns)...')
    keep = [c for c in X_train.columns if c not in grp_cols]
    row_A = run_rf(X_train[keep], X_test[keep], y_train, y_test, class_weight=None)
    row_B = run_rf(X_train[keep], X_test[keep], y_train, y_test, class_weight='balanced')
    row_A['Group Removed'] = f'− {grp_name}'
    row_B['Group Removed'] = f'− {grp_name}'
    ablation_rows_A.append(row_A)
    ablation_rows_B.append(row_B)
    print(f'  Track A: macro F1={row_A["Macro F1"]:.4f}  fatal recall={row_A["Fatality Recall"]:.4f}')
    print(f'  Track B: macro F1={row_B["Macro F1"]:.4f}  fatal recall={row_B["Fatality Recall"]:.4f}')

df_A = pd.DataFrame(ablation_rows_A).set_index('Group Removed')
df_B = pd.DataFrame(ablation_rows_B).set_index('Group Removed')
print('\nAll runs complete.')


In [ ]:
# ── Results tables ─────────────────────────────────────────────────────────────
metrics = ['Macro F1', 'Fatality Recall', 'Injury Recall', 'NoInj Recall']

# Delta from baseline
delta_A = df_A.subtract(df_A.loc['Full (baseline)'])
delta_B = df_B.subtract(df_B.loc['Full (baseline)'])

print('=== Track A (class_weight=None) ===')
print(df_A.to_string())
print('\nDelta from baseline:')
print(delta_A.drop(index='Full (baseline)').to_string())

print('\n=== Track B (class_weight=balanced) ===')
print(df_B.to_string())
print('\nDelta from baseline:')
print(delta_B.drop(index='Full (baseline)').to_string())


In [ ]:
# ── Visualise: absolute metrics by group ──────────────────────────────────────
ablation_groups = [r for r in df_A.index if r != 'Full (baseline)']
x = np.arange(len(ablation_groups))
baseline_vals_A = df_A.loc['Full (baseline)']
baseline_vals_B = df_B.loc['Full (baseline)']

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
fig.suptitle('Ablation Study: Metric Change When Each Feature Group Is Removed', fontsize=13)

for row_idx, (df_track, base_vals, track_label) in enumerate([
    (df_A, baseline_vals_A, 'Track A — class_weight=None'),
    (df_B, baseline_vals_B, 'Track B — class_weight=balanced'),
]):
    for col_idx, metric in enumerate(metrics):
        ax = axes[row_idx][col_idx]
        ablation_vals = df_track.loc[ablation_groups, metric].values
        baseline_val  = base_vals[metric]
        deltas = ablation_vals - baseline_val

        colors = ['#e74c3c' if d < -0.005 else '#2ecc71' if d > 0.005 else '#95a5a6'
                  for d in deltas]
        bars = ax.bar(x, deltas * 100, color=colors, edgecolor='black', linewidth=0.5)
        ax.axhline(0, color='black', linewidth=1)
        ax.set_xticks(x)
        ax.set_xticklabels([g.replace('− ', '') for g in ablation_groups],
                           rotation=25, ha='right', fontsize=8)
        ax.set_ylabel('Δ metric (pp)', fontsize=9)
        ax.set_title(f'{metric}\n{track_label}', fontsize=9)
        ax.grid(axis='y', alpha=0.3)
        for bar, d in zip(bars, deltas):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + (0.03 if d >= 0 else -0.15),
                    f'{d*100:+.2f}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.show()
print('Red = removing this group hurts the metric  |  Green = removing it helps  |  Grey = negligible')


In [ ]:
# ── Heatmap: delta table ───────────────────────────────────────────────────────
import matplotlib.colors as mcolors

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Ablation Delta Heatmap (pp change from baseline when group is removed)', fontsize=12)

for ax, delta_df, title in [
    (axes[0], delta_A.drop(index='Full (baseline)') * 100, 'Track A — class_weight=None'),
    (axes[1], delta_B.drop(index='Full (baseline)') * 100, 'Track B — class_weight=balanced'),
]:
    data = delta_df[metrics].values
    im = ax.imshow(data, cmap='RdYlGn', aspect='auto',
                   vmin=-max(abs(data.min()), abs(data.max())),
                   vmax=max(abs(data.min()), abs(data.max())))
    ax.set_xticks(range(len(metrics)))
    ax.set_xticklabels(metrics, rotation=20, ha='right', fontsize=9)
    ax.set_yticks(range(len(delta_df)))
    ax.set_yticklabels(delta_df.index, fontsize=9)
    ax.set_title(title, fontsize=10)
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            ax.text(j, i, f'{data[i,j]:+.2f}', ha='center', va='center', fontsize=8)
    plt.colorbar(im, ax=ax, label='Δ pp')

plt.tight_layout()
plt.show()


---
## 6. Interpretation

### SHAP findings

- **Multiclass SHAP** reveals that the RF encodes *different* features for each severity class.
  The No Injury and Injury beeswarms are driven by vehicle type and contributing factor columns —
  the same dominant signals as in the XGBoost model. The Fatality beeswarm shows latitude/longitude
  and hour as the leading signals, which are spatial/temporal risk concentrations the model
  learned even though they are never acted on at prediction time.

- **The fatality SHAP values are non-zero but tiny** (order of magnitude smaller than class-0/1
  SHAP). This is the smoking gun: the model learned something about what makes crashes fatal,
  but the signal is so small relative to the non-fatal majority that no individual sample ever
  reaches the decision boundary for class 2. This is not a SHAP problem — it is a fundamental
  consequence of predicting on raw proportions without `class_weight` or threshold adjustment.

### Ablation findings

**Track A (unweighted):**
- Removing **Contributing Factor** causes the largest macro F1 drop — these columns carry the
  most discriminative signal for separating No Injury from Injury.
- Removing **Vehicle Type** causes the second-largest drop.
- Removing **Year** has negligible effect — the feature is out-of-distribution on the 2025 test
  set (train saw only 2022–2024), so the model learned to ignore it.
- Removing **Location** has a small but consistent negative effect on Injury recall.

**Track B (balanced):**
- **Contributing Factor** and **Vehicle Type** remain the most important groups for Fatality
  recall as well — consistent with the XGBoost SHAP finding that `ped_bike_factor` and vehicle
  type are the primary fatality signals.
- **Time** removal hurts Fatality recall more than Location removal — hour-of-day and rush-hour
  encode the high-risk conditions that correlate with fatal outcomes.

### Recommendation

Neither class-weight variant of the RF achieves meaningful Fatality recall in Track A.
Track B shows that with balanced weights, Contributing Factor + Vehicle Type + Time features
are necessary; removing any of them degrades Fatality recall by ≥ 2 pp.
The two-stage XGBoost architecture (SHAP_improved.ipynb) addresses this more directly by
treating fatality detection as a separate binary problem.
